In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import neurokit2 as nk
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, filtfilt, find_peaks

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr
from scipy.integrate import trapezoid
import scipy.stats as stats
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
import scipy.signal as scisig
from scipy.signal import resample
import json
import ast

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
import onnx

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    matthews_corrcoef, roc_auc_score
)

import warnings
warnings.filterwarnings(
    "ignore",
    module="neurokit2"
)

np.set_printoptions(precision=3, suppress=True)

In [3]:
# ============================ Data Feature Parameters =========================================
DATA_PATH = "../data"
LABEL_WINDOW = 2
INPUT_WINDOW = 60
SAMPLE_FS = {'ACC': 32, 'BVP': 64, 'TEMP': 4, 'LABEL': 700, 'EDA': 4}
USE_FREQ_HRV = False

SUBJECT_IDS = (
    [f"S{i}" for i in range(2, 12)] +
    [f"S{i}" for i in range(13, 18)]
)

def downsample_label(x, fs_high, fs_low):
    x = np.asarray(x[max(0, (INPUT_WINDOW-LABEL_WINDOW) * fs_high):])
    T = len(x) / fs_high
    n_out = int(round(T * fs_low))

    # time stamps for input samples
    t = np.arange(len(x)) / fs_high

    # bin edges for output samples (uniform in time)
    edges = np.linspace(0, T, n_out + 1)

    y = np.empty(n_out, dtype=int)
    av = np.empty((n_out, 3), dtype=int)
    for i in range(n_out):
        mask = (t >= edges[i]) & (t < edges[i+1])

        if mask.any():
            l = np.bincount(x[mask], minlength=8)[1:5]
            arousal = 1 if l[0] + l[3] < l[1] + l[2] else 0
            valence = 1 if l[0] + l[1] < l[2] + l[3] else 0
            fourclass = 0
            if arousal == 0 and valence == 0:
                fourclass = 0
            elif arousal == 0 and valence == 1:
                fourclass = 1
            elif arousal == 1 and valence == 0:
                fourclass = 2
            elif arousal == 1 and valence ==1:
                fourclass = 3
            y[i] = l.argmax() if l.argmax() != 3 else 0
            av[i] = np.array([arousal, valence, fourclass])
        else:
            # rare edge case if rounding creates an empty bin
            y[i] = y[i-1] if i > 0 else x[0]
            av[i] = av[i-1] if i > 0 else np.array([0, 0, 0])
    return y, av

def upsample(low_signal, high_signal, fs_high, fs_low):
    upsample_factor = fs_high / fs_low

    # Compute target length
    target_len = len(high_signal)

    # Repeat each low-frequency sample
    indices = np.floor(np.arange(target_len) / upsample_factor).astype(int)
    indices = np.clip(indices, 0, len(low_signal) - 1)

    return low_signal[indices]

def filterSignalFIR(data, cutoff=0.4, numtaps=64):
    f = cutoff / (32 / 2.0)
    FIR_coeff = scisig.firwin(numtaps, f)
    return scisig.lfilter(FIR_coeff, 1, data)

def dom_nonzero_freq(signal, fs_hz):
    """
    Returns most dominant non-zero frequency via fft
    """
    T, K, C = signal.shape
    y = np.fft.rfft(signal, axis = 0)
    yf = np.fft.rfftfreq(T, 1/fs_hz)
    mag = np.abs(y)
    mag[0, :, :] = 0
    idx = np.argmax(mag, axis = 0)
    return yf[idx]

def eda_features(eda, fs, corr_method="pearson"):
    """
    Compute SCR/SCL features from an EDA signal.

    Parameters
    ----------
    eda : array-like
        Raw EDA signal (skin conductance).
    fs : float
        Sampling frequency in Hz.
    corr_method : str
        "pearson" or "spearman" for SCL-time correlation.

    Returns
    -------
    features : dict
        Dictionary of computed features.
    signals : pd.DataFrame
        Processed signal dataframe from neurokit2.
    info : dict
        Event metadata returned by neurokit2.
    """
    eda = np.asarray(eda, dtype=float)

    # 1) Clean + decompose + detect SCR peaks
    # signals columns typically include:
    # EDA_Clean, EDA_Tonic (SCL), EDA_Phasic (SCR), SCR_Onsets, SCR_Peaks, SCR_Recovery, ...
    signals, info = nk.eda_process(eda, sampling_rate=fs)

    scl = signals["EDA_Tonic"].to_numpy()   # tonic = SCL
    scr = signals["EDA_Phasic"].to_numpy()  # phasic = SCR

    # Time vector
    t = np.arange(len(eda)) / fs

    # 2) Correlation between SCL and time
    if corr_method.lower() == "pearson":
        scl_time_corr, scl_time_corr_p = pearsonr(t, scl)
    elif corr_method.lower() == "spearman":
        scl_time_corr, scl_time_corr_p = spearmanr(t, scl)
    else:
        raise ValueError("corr_method must be 'pearson' or 'spearman'")

    # 3) Get SCR event indices
    # NeuroKit2 stores indices in info, but keys may vary slightly depending on version.
    onsets = np.array(info.get("SCR_Onsets", []), dtype=float)
    peaks = np.array(info.get("SCR_Peaks", []), dtype=float)
    recovery = np.array(info.get("SCR_Recovery", []), dtype=float)

    # Remove NaNs / invalid values and cast to int
    onsets = onsets[np.isfinite(onsets)].astype(int)
    peaks = peaks[np.isfinite(peaks)].astype(int)
    recovery = recovery[np.isfinite(recovery)].astype(int)

    # Align events robustly (onset -> peak -> recovery)
    # We'll create matched segments where onset < peak < recovery if possible.
    segments = []
    used_recovery = set()

    for peak in peaks:
        onset_candidates = onsets[onsets < peak]
        if len(onset_candidates) == 0:
            continue
        onset = onset_candidates[-1]  # nearest onset before peak

        rec_candidates = recovery[(recovery > peak)]
        rec_candidates = [r for r in rec_candidates if r not in used_recovery]
        if len(rec_candidates) == 0:
            # If no recovery marker, estimate end at next time SCR returns near local baseline
            # Simple fallback: use peak + 4s (capped to signal length)
            rec = min(len(scr) - 1, int(peak + 4 * fs))
        else:
            rec = rec_candidates[0]
            used_recovery.add(rec)

        if onset < peak < rec:
            segments.append((onset, peak, rec))

    # 4) Number of SCR segments
    n_scr_segments = len(segments)

    # 5) SCR startle magnitudes and response durations
    # Startle magnitude here = peak amplitude relative to onset baseline (phasic component)
    # Duration = recovery - onset (seconds)
    startle_magnitudes = []
    response_durations = []
    scr_auc_list = []

    for onset, peak, rec in segments:
        magnitude = scr[peak] - scr[onset]
        duration = (rec - onset) / fs

        # Area under identified SCR segment (above onset baseline)
        segment_y = scr[onset:rec + 1] - scr[onset]
        # Optional: clip negative values so only positive response contributes
        segment_y = np.clip(segment_y, 0, None)
        segment_t = t[onset:rec + 1]
        auc = trapezoid(segment_y, segment_t)

        startle_magnitudes.append(magnitude)
        response_durations.append(duration)
        scr_auc_list.append(auc)

    startle_magnitudes = np.array(startle_magnitudes, dtype=float)
    response_durations = np.array(response_durations, dtype=float)
    scr_auc_list = np.array(scr_auc_list, dtype=float)

    sum_scr_startle_magnitudes = float(np.nansum(startle_magnitudes)) if len(startle_magnitudes) else 0.0
    sum_response_durations = float(np.nansum(response_durations)) if len(response_durations) else 0.0
    area_under_identified_scr = float(np.nansum(scr_auc_list)) if len(scr_auc_list) else 0.0

    features = {
        "EDA_mean": np.mean(eda),
        "EDA_std": np.std(eda),
        "EDA_max": max(eda),
        "EDA_min": min(eda),
        "EDA_slope": np.polyfit(np.arange(len(eda)), eda, 1)[0],
        "EDA_range": max(eda) - min(eda),

        # Whole-signal tonic/phasic summaries
        "SCL_mean": float(np.nanmean(scl)),
        "SCL_std": float(np.nanstd(scl)),
        "SCR_mean": float(np.nanmean(scr)),
        "SCR_std": float(np.nanstd(scr)),

        # Correlation
        "SCL_time_corr": float(scl_time_corr),
        "SCL_time_corr_pvalue": float(scl_time_corr_p),

        # Event-based SCR features
        "num_SCR_segments": int(n_scr_segments),
        "sum_SCR_startle_magnitudes": sum_scr_startle_magnitudes,
        "sum_response_durations_sec": sum_response_durations,
        "area_under_identified_SCR": area_under_identified_scr,

        # Optional per-event stats
        "mean_SCR_startle_magnitude": float(np.nanmean(startle_magnitudes)) if len(startle_magnitudes) else 0.0,
        "mean_response_duration_sec": float(np.nanmean(response_durations)) if len(response_durations) else 0.0,
        "mean_SCR_auc": float(np.nanmean(scr_auc_list)) if len(scr_auc_list) else 0.0,
    }

    return features, signals, info

In [3]:
# =========================== Data processing: WESAD Table ===================================================
def load_subject(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")
    
def load_survey(path):
    with open(path,"r") as f:
        return f.readlines()
    
x = []
y = []
data_subject_indices = []
WESAD_path = DATA_PATH + "/WESAD"
for sid in SUBJECT_IDS:
    print(sid)
    data_subject_indices.append(len(x))
    print(data_subject_indices)
    subject = load_subject(f"{WESAD_path}/{sid}/{sid}.pkl")
    survey = load_survey(f"{WESAD_path}/{sid}/{sid}_readme.txt")

    labels = np.array(subject['label'])
    labels = downsample_label(labels, SAMPLE_FS['LABEL'], 1.0/LABEL_WINDOW)

    temp = np.array(subject['signal']['wrist']['TEMP'])

    bvp = np.array(subject['signal']['wrist']['BVP'])
    bvp_signals, bvp_info = nk.ppg_process(bvp, sampling_rate=SAMPLE_FS['BVP'])

    acc = np.array(subject['signal']['wrist']['ACC'])
    #acc = filterSignalFIR(acc)

    eda = np.array(subject['signal']['wrist']['EDA']).squeeze(1)

    survey_vec = [int(survey[i].split(" ")[-1]) for i in range(1, 4)]

    if survey[4].split(" ")[-1] == "male\n":
        survey_vec += [1, 0]
    elif survey[4].split(" ")[-1] == "female\n":
        survey_vec += [0, 1]
    else:
        raise Exception("Unexpected gender")
    
    if survey[5].split(" ")[-1] == "right\n":
        survey_vec += [1, 0]
    elif survey[5].split(" ")[-1] == "left\n":
        survey_vec += [0, 1]
    else:
        raise Exception("Unexpected dominant hand")
    
    for i in range(8, 14):
        response = survey[i].split(" ")[-1]
        if response == "YES\n":
            survey_vec += [1, 0]
        elif response == "NO\n":
            survey_vec += [0, 1]
        else:
            raise Exception("Unexpected yes or no")
    print(survey_vec)

    for start in range(0, len(labels)):
        temp_s = temp[start * LABEL_WINDOW * SAMPLE_FS['TEMP']: start * LABEL_WINDOW * SAMPLE_FS['TEMP'] + INPUT_WINDOW * SAMPLE_FS['TEMP']]
        bvp_s = bvp[start * LABEL_WINDOW * SAMPLE_FS['BVP']: start * LABEL_WINDOW * SAMPLE_FS['BVP'] + INPUT_WINDOW * SAMPLE_FS['BVP']]
        peaks_s = np.array(bvp_signals['PPG_Peaks'][start * LABEL_WINDOW * SAMPLE_FS['BVP']: start * LABEL_WINDOW * SAMPLE_FS['BVP'] + INPUT_WINDOW * SAMPLE_FS['BVP']])
        if np.count_nonzero(peaks_s) <= 3 or len(bvp_s) != INPUT_WINDOW * SAMPLE_FS['BVP']:
            continue
        acc_s = acc[start * LABEL_WINDOW * SAMPLE_FS['ACC']: start * LABEL_WINDOW * SAMPLE_FS['ACC'] + INPUT_WINDOW * SAMPLE_FS['ACC']]
        acc_s = acc_s.reshape(160, 12, 3)
        eda_s = eda[start * LABEL_WINDOW * SAMPLE_FS['EDA']: start * LABEL_WINDOW * SAMPLE_FS['EDA'] + INPUT_WINDOW * SAMPLE_FS['EDA']]
        label_s = labels[start]

        acc_mean = np.mean(acc_s, axis = 0)
        acc_std = np.std(acc_s, axis = 0)
        acc_sum = np.trapezoid(np.abs(acc_s), axis = 0)
        acc_peak_freq = dom_nonzero_freq(acc_s, SAMPLE_FS['ACC'])
        acc_data = np.concatenate([acc_mean.flatten(), acc_std.flatten(), acc_sum.flatten(), acc_peak_freq.flatten()])

        temp_mean = np.mean(temp_s)
        temp_std = np.std(temp_s)
        temp_min = np.min(temp_s)
        temp_max = np.max(temp_s)
        temp_range = temp_max - temp_min
        temp_slope = np.polyfit(np.arange(len(temp_s)), temp_s, 1)[0][0]
        temp_data = [temp_mean, temp_std, temp_min, temp_max, temp_range, temp_slope]
        
        hrv = nk.hrv_time(peaks_s, sampling_rate=SAMPLE_FS['BVP'], show=False) if not USE_FREQ_HRV else nk.hrv(peaks_s, sampling_rate=SAMPLE_FS['BVP'], show=False)
        hr_mean = hrv['HRV_MeanNN']
        hr_std = hrv['HRV_SDNN']
        hrv_rmssd = hrv['HRV_RMSSD']
        hrv_pNN50 = hrv['HRV_pNN50']
        hrv_tinn = hrv['HRV_TINN']
        hrv_data = [hr_mean, hr_std, hrv_rmssd, hrv_pNN50, hrv_tinn]

        eda_feature, _, _ = eda_features(eda_s, SAMPLE_FS['EDA'])
        eda_data = [eda_feature["EDA_mean"], eda_feature["EDA_std"], eda_feature["EDA_max"], eda_feature["EDA_min"], eda_feature["EDA_slope"], eda_feature["EDA_range"], 
                    eda_feature["SCL_mean"], eda_feature["SCL_std"], eda_feature["SCR_mean"], eda_feature["SCR_std"], eda_feature["SCL_time_corr"], eda_feature["num_SCR_segments"],
                    eda_feature["sum_SCR_startle_magnitudes"], eda_feature["sum_response_durations_sec"], eda_feature["area_under_identified_SCR"]]

        if USE_FREQ_HRV:
            hrv_data.append(hrv['HRV_ULF'])
            hrv_data.append(hrv['HRV_LF'])
            hrv_data.append(hrv['HRV_HF'])
            hrv_data.append(hrv['HRV_VHF'])
            hrv_data.append(hrv['HRV_LFHF'])
            hrv_data.append(hrv['HRV_LFn'])
            hrv_data.append(hrv['HRV_HFn'])

        x_s = np.array(np.concatenate([acc_data, temp_data, np.array(hrv_data).squeeze(1), eda_data, np.array(survey_vec)]))

        x.append(x_s)
        y.append(label_s)
data_subject_indices.append(len(x))
print(data_subject_indices)

x = np.array(x)
y = np.array(y)
print(x.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2, shuffle = True, stratify=y)
    


S2
[0]


/tmp/ipykernel_100736/750776912.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="latin1")


KeyboardInterrupt: 

In [51]:
# =========================== Data processing: EmoWear Table ===================================================

EmoWear_Path = DATA_PATH + "/EmoWear/csv"
subjects = [name for name in os.listdir(EmoWear_Path)if os.path.isdir(os.path.join(EmoWear_Path, name))]

survey = pd.read_csv(DATA_PATH + "/EmoWear/questionnaire.csv")
survey.drop(34,axis=0,inplace=True)
survey.drop(columns=['Code', 'ID'], inplace=True)
survey_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False), 
         ['Gender', 'Handedness','Vision',"Vision aid", 'Education','Alcohol consumption', 'Coffee consumption',
          'Tea consumption', 'Tobacco consumption', 'Other drug/medication consumption', 'Level of Alertness',
          'Physical/psychiatric syndroms'])
    ],
    remainder='passthrough'
)
survey_encoded = survey_preprocessor.fit_transform(survey)

x = []
y = []
data_subject_indices = []
for sid in subjects:
    print(sid)
    path = EmoWear_Path + f"/{sid}/"
    data_subject_indices.append(len(x))

    acc = pd.read_csv(path + "signals-e4-acc.csv", header=0).to_numpy()
    temp = pd.read_csv(path + "signals-e4-skt.csv", header=0).to_numpy()
    eda = pd.read_csv(path + "signals-e4-eda.csv", header=0).to_numpy()
    bvp = pd.read_csv(path + "signals-e4-bvp.csv", header=0).to_numpy()
    bvp_signals, bvp_info = nk.ppg_process(bvp[:,1], sampling_rate=SAMPLE_FS['BVP'])

    label = pd.read_csv(path + "surveys.csv", header=0).to_numpy()
    label_t = pd.read_csv(path + "markers-phase2.csv", header=0).to_numpy()

    for i,seq in enumerate(label_t):
        start_time = seq[2]
        end_time = seq[-1]
        num_step = int((end_time - start_time - INPUT_WINDOW) // LABEL_WINDOW + 1)

        acc_start = np.searchsorted(acc[:, 0], start_time)
        temp_start = np.searchsorted(temp[:, 0], start_time)
        eda_start = np.searchsorted(eda[:, 0], start_time)
        bvp_start = np.searchsorted(bvp[:, 0], start_time)
        label_s = label[i, 2:]
        for t in range(num_step):
            acc_s = acc[acc_start + LABEL_WINDOW * t * SAMPLE_FS['ACC']:
                           acc_start + LABEL_WINDOW * t * SAMPLE_FS['ACC'] + SAMPLE_FS['ACC'] * INPUT_WINDOW, 1:]
            acc_s = acc_s.reshape(160, 12, 3)
            temp_s = temp[temp_start + LABEL_WINDOW * t * SAMPLE_FS['TEMP']:
                           temp_start + LABEL_WINDOW * t * SAMPLE_FS['TEMP'] + SAMPLE_FS['TEMP'] * INPUT_WINDOW, 1]
            bvp_s = bvp[bvp_start + LABEL_WINDOW * t * SAMPLE_FS['BVP']:
                           bvp_start + LABEL_WINDOW * t * SAMPLE_FS['BVP'] + SAMPLE_FS['BVP'] * INPUT_WINDOW, 1]
            peaks_s = np.array(bvp_signals['PPG_Peaks'][bvp_start + LABEL_WINDOW * t * SAMPLE_FS['BVP']:
                           bvp_start + LABEL_WINDOW * t * SAMPLE_FS['BVP'] + SAMPLE_FS['BVP'] * INPUT_WINDOW])
            eda_s = eda[eda_start + LABEL_WINDOW * t * SAMPLE_FS['EDA']:
                        eda_start + LABEL_WINDOW * t * SAMPLE_FS['EDA'] + SAMPLE_FS['EDA'] * INPUT_WINDOW, 1]

            if np.count_nonzero(peaks_s) <= 3 or len(bvp_s) != INPUT_WINDOW * SAMPLE_FS['BVP']:
                continue

            acc_mean = np.mean(acc_s, axis = 0)
            acc_std = np.std(acc_s, axis = 0)
            acc_sum = np.trapezoid(np.abs(acc_s), axis = 0)
            acc_peak_freq = dom_nonzero_freq(acc_s, SAMPLE_FS['ACC'])
            acc_data = np.concatenate([acc_mean.flatten(), acc_std.flatten(), acc_sum.flatten(), acc_peak_freq.flatten()])

            temp_mean = np.mean(temp_s)
            temp_std = np.std(temp_s)
            temp_min = np.min(temp_s)
            temp_max = np.max(temp_s)
            temp_range = temp_max - temp_min
            temp_slope = np.polyfit(np.arange(len(temp_s)), temp_s, 1)[0]
            temp_data = [temp_mean, temp_std, temp_min, temp_max, temp_range, temp_slope]
            
            hrv = nk.hrv_time(peaks_s, sampling_rate=SAMPLE_FS['BVP'], show=False) if not USE_FREQ_HRV else nk.hrv(peaks_s, sampling_rate=SAMPLE_FS['BVP'], show=False)
            hr_mean = hrv['HRV_MeanNN']
            hr_std = hrv['HRV_SDNN']
            hrv_rmssd = hrv['HRV_RMSSD']
            hrv_pNN50 = hrv['HRV_pNN50']
            hrv_tinn = hrv['HRV_TINN']
            hrv_data = [hr_mean, hr_std, hrv_rmssd, hrv_pNN50, hrv_tinn]

            eda_feature, _, _ = eda_features(eda_s, SAMPLE_FS['EDA'])
            eda_data = [eda_feature["EDA_mean"], eda_feature["EDA_std"], eda_feature["EDA_max"], eda_feature["EDA_min"], eda_feature["EDA_slope"], eda_feature["EDA_range"], 
                        eda_feature["SCL_mean"], eda_feature["SCL_std"], eda_feature["SCR_mean"], eda_feature["SCR_std"], eda_feature["SCL_time_corr"], eda_feature["num_SCR_segments"],
                        eda_feature["sum_SCR_startle_magnitudes"], eda_feature["sum_response_durations_sec"], eda_feature["area_under_identified_SCR"]]

            if USE_FREQ_HRV:
                hrv_data.append(hrv['HRV_ULF'])
                hrv_data.append(hrv['HRV_LF'])
                hrv_data.append(hrv['HRV_HF'])
                hrv_data.append(hrv['HRV_VHF'])
                hrv_data.append(hrv['HRV_LFHF'])
                hrv_data.append(hrv['HRV_LFn'])
                hrv_data.append(hrv['HRV_HFn'])

            x_s = np.array(np.concatenate([acc_data, temp_data, np.array(hrv_data).squeeze(1), eda_data, np.array(survey_encoded[i])]))

            x.append(x_s)
            y.append(label_s)

data_subject_indices.append(len(x))
print(data_subject_indices)
x = np.array(x)
y = np.array(y)
print(x.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2, shuffle = True, stratify=y)

27-9VUW
17-9UTL
16-9USC
01-9TZK
24-9VL3
31-9VZJ
23-9VIV
33-9VZR
39-9W6G
29-9VXR
14-9UP0
47-9WCU
34-9W1R
26-9VTJ
28-9VXK
04-9U66
07-9UAD
13-9UOF
21-9V5J
43-9W91
06-9UA7
25-9VNK
46-9WCE
08-9UB2
45-9WBS
12-9UN3
40-9W74
20-9V52
44-9WAY
42-9W7R
49-9WEW
41-9W7K
22-9V5N
02-9TZO
32-9VZN
03-9U4C
19-9UY7
09-9UDP
36-9W4C
38-9W6A
11-9UIZ
18-9UVM
10-9UHM
30-9VYD
48-9WDR
37-9W4X
05-9U8B
15-9UR2
[0, 1161, 2255, 3201, 4325, 5370, 6495, 7592, 8619, 9751, 10642, 11204, 12170, 12784, 13778, 15081, 16319, 17462, 18612, 19938, 20796, 21710, 22856, 23762, 24892, 25916, 26994, 28087, 29079, 30091, 31043, 32227, 33424, 34534, 35711, 36837, 37892, 38814, 39350, 40353, 41334, 42219, 43345, 44314, 45395, 46454, 47429, 48580, 49495]
(49495, 254)
(49495, 5)


In [76]:
# =========================== Data processing: Engagnition Table ===================================================
survey_data = pd.DataFrame([
    ["P01", 11, "M", "ASD"],
    ["P02", 9,  "M", "ASD, ID"],
    ["P03", 11, "F", "ASD"],
    ["P04", 10, "F", "ASD"],
    ["P05", 10, "F", "ASD, ID"],
    ["P06", 10, "F", "ASD"],
    ["P07", 4,  "M", "ASD"],
    ["P08", 13, "M", "ASD"],
    ["P09", 10, "M", "ASD, ID"],
    ["P10", 10, "M", "ASD"],
    ["P11", 7,  "M", "ADHD"],
    ["P12", 11, "M", "ASD"],
    ["P13", 12, "M", "ASD, ID"],
    ["P14", 8,  "F", "ASD, ID"],
    ["P15", 10, "M", "ASD"],
    ["P16", 9,  "M", "ASD, ID"],
    ["P17", 11, "M", "ASD, ID"],
    ["P18", 12, "M", "ASD"],
    ["P19", 6,  "F", "ASD, ID"],
    ["P20", 11, "M", "ASD"],
    ["P21", 9,  "M", "ASD"],
    ["P22", 10, "M", "ASD"],
    ["P23", 11, "F", "ASD"],
    ["P24", 10, "F", "ASD"],
    ["P25", 10, "F", "ASD, ID"],
    ["P26", 10, "F", "ASD"],
    ["P27", 11, "M", "ASD, ID"],
    ["P28", 6,  "F", "ASD, ID"],
    ["P29", 13, "M", "ASD"],
    ["P30", 10, "M", "ASD, ID"],
    ["P31", 10, "M", "ASD"],
    ["P32", 7,  "M", "ADHD"],
    ["P33", 11, "M", "ASD"],
    ["P34", 12, "M", "ASD, ID"],
    ["P35", 8,  "F", "ASD, ID"],
    ["P36", 9,  "M", "ASD, ID"],
    ["P37", 12, "M", "ASD"],
    ["P38", 4,  "M", "ASD"],
    ["P39", 11, "M", "ASD"],
    ["P40", 9,  "M", "ASD, ID"],
    ["P41", 11, "F", "ASD"],
    ["P42", 10, "F", "ASD"],
    ["P43", 10, "F", "ASD, ID"],
    ["P44", 10, "F", "ASD"],
    ["P45", 10, "M", "ASD, ID"],
    ["P46", 10, "M", "ASD"],
    ["P47", 8,  "M", "ASD, ID"],
    ["P48", 13, "M", "ASD"],
    ["P49", 16, "M", "ASD"],
    ["P50", 12, "M", "ASD"],
    ["P51", 12, "M", "ASD, ID"],
    ["P52", 10, "M", "ASD"],
    ["P53", 10, "M", "ASD"],
    ["P54", 10, "M", "ASD, ID"],
    ["P55", 11, "M", "ASD, ID"],
    ["P56", 7,  "M", "ADHD"],
    ["P57", 8,  "F", "ASD, ID"],
], columns = ['id', 'age', 'gender', 'diagnosis'])
survey_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False), 
         ['gender', 'diagnosis'])
    ],
    remainder='passthrough'
)
survey_encoded = survey_preprocessor.fit_transform(survey_data)

Engagnition_Path = DATA_PATH + "/Engagnition/"
Folders = ["Baseline condition", "HPE condition", "LPE condition"]
Subjects_Group = [[f"P{str(i).zfill(2)}" for i in range(1,20)], [f"P{str(i).zfill(2)}" for i in range(39,58)], [f"P{str(i).zfill(2)}" for i in range(20, 39)]]

x = []
y = []
data_subject_indices = []

for condition, (f, subjects) in enumerate(zip(Folders, Subjects_Group)):
    for s in subjects:
        print(s)
        data_subject_indices.append(len(x))
        path = Engagnition_Path + f"{f}/{s}/"

        acc = pd.read_csv(path + "E4AccData.csv", header=0).to_numpy()
        temp = pd.read_csv(path + "E4TmpData.csv", header=0).to_numpy()
        eda = pd.read_csv(path + "E4GsrData.csv", header=0).to_numpy()

        engagement = np.full((1,2), -5)
        gaze = np.full((1,2), -5)
        performance = np.full((1,2), -5)
        if f != "Baseline condition":
            engagement = pd.read_csv(path + "EngagementData.csv", header=0).to_numpy()
            gaze = pd.read_csv(path + "GazeData.csv", header=0).to_numpy()
            performance = pd.read_csv(path + "PerformanceData.csv", header=0).to_numpy()

        start = max(acc[0,0], temp[0,0], eda[0,0])
        end = min(acc[-1,0], temp[-1,0], eda[-1,0])
        t = start + INPUT_WINDOW

        while t <= end:
            acc_mask = (acc[:,0] > t - INPUT_WINDOW) & (acc[:,0] <= t)
            temp_mask = (temp[:,0] > t - INPUT_WINDOW) & (temp[:,0] <= t)
            eda_mask = (eda[:,0] > t - INPUT_WINDOW) & (eda[:,0] <= t)
            engagement_mask = (engagement[:,0] > t - LABEL_WINDOW) & (engagement[:,0] <= t)
            gaze_mask = (gaze[:,0] > t - LABEL_WINDOW) & (gaze[:,0] <= t)
            performance_mask = (performance[:,0] > t - LABEL_WINDOW) & (performance[:,0] <= t)

            acc_s = acc[acc_mask, 2:-1]
            acc_s = np.expand_dims(acc_s, axis = 1)
            temp_s = temp[temp_mask, 2]
            eda_s = eda[eda_mask, 2]

            eda_feature, _, _ = eda_features(eda_s, SAMPLE_FS['EDA'])
            eda_data = [eda_feature["EDA_mean"], eda_feature["EDA_std"], eda_feature["EDA_max"], eda_feature["EDA_min"], eda_feature["EDA_slope"], eda_feature["EDA_range"], 
                        eda_feature["SCL_mean"], eda_feature["SCL_std"], eda_feature["SCR_mean"], eda_feature["SCR_std"], eda_feature["SCL_time_corr"], eda_feature["num_SCR_segments"],
                        eda_feature["sum_SCR_startle_magnitudes"], eda_feature["sum_response_durations_sec"], eda_feature["area_under_identified_SCR"]]
            acc_mean = np.mean(acc_s, axis = 0)
            acc_std = np.std(acc_s, axis = 0)
            acc_sum = np.trapezoid(np.abs(acc_s), axis = 0)
            acc_peak_freq = dom_nonzero_freq(acc_s, SAMPLE_FS['ACC'])
            acc_data = np.concatenate([acc_mean.flatten(), acc_std.flatten(), acc_sum.flatten(), acc_peak_freq.flatten()])

            temp_mean = np.mean(temp_s)
            temp_std = np.std(temp_s)
            temp_min = np.min(temp_s)
            temp_max = np.max(temp_s)
            temp_range = temp_max - temp_min
            temp_slope = np.polyfit(np.arange(len(temp_s)), temp_s, 1)[0]
            temp_data = [temp_mean, temp_std, temp_min, temp_max, temp_range, temp_slope]
            

            engagement_s = engagement[engagement_mask, 1]
            gaze_s = gaze[gaze_mask, 1]
            performance_s = performance[performance_mask, 1]

            label_e_i = np.bincount(engagement_s.astype(np.int32), minlength=3).argmax() if len(engagement_s) > 0 else -1
            label_g_i = np.bincount(gaze_s.astype(np.int32), minlength=2).argmax() if len(gaze_s) > 0 else -1
            label_p_i = np.bincount(performance_s.astype(np.int32), minlength=2).argmax() if len(performance_s) > 0 else -1

            label_e = np.zeros(4)
            label_e[label_e_i] = 1
            label_g = np.zeros(3)
            label_g[label_g_i] = 1
            label_p = np.zeros(3)
            label_p[label_p_i] = 1
            label_c = np.zeros(3)
            label_c[condition] = 1

            label = np.concatenate([label_e, label_g, label_p, label_c])
            x_s = np.array(np.concatenate([acc_data, temp_data, eda_data, np.array([surv for surv in survey_encoded[int(s[1:])-1] if surv != s])]))

            x.append(x_s)
            y.append(label)

            t += LABEL_WINDOW

data_subject_indices.append(len(x))
print(data_subject_indices)
x = np.array(x)
y = np.array(y)
print(x.shape)
print(y.shape)
        


P01
P02
P03
P04
P05
P06
P07
P08
P09
P10
P11
P12
P13
P14
P15
P16
P17
P18
P19
P39
P40
P41
P42
P43
P44
P45
P46
P47
P48
P49
P50
P51
P52
P53
P54
P55
P56
P57
P20
P21
P22
P23
P24
P25
P26
P27
P28
P29
P30
P31
P32
P33
P34
P35
P36
P37
P38
[0, 59, 118, 177, 237, 303, 376, 443, 505, 566, 627, 748, 814, 883, 944, 1020, 1105, 1192, 1282, 1347, 1511, 1801, 1978, 2173, 2301, 2616, 2788, 2907, 3061, 3281, 3403, 3499, 3572, 3672, 3928, 4082, 4311, 4463, 4843, 4887, 4937, 5029, 5062, 5108, 5151, 5349, 5449, 5568, 5568, 5626, 5662, 5702, 5727, 5755, 5946, 6048, 6069, 6228]
(6228, 39)
(6228, 13)


In [ ]:
# =========================== Data processing: WESAD Table For FEEL ===================================================
def load_subject(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")
    
def load_survey(path):
    with open(path,"r") as f:
        return f.readlines()
    
csv_data = pd.DataFrame()
data_subject_indices = []
WESAD_path = DATA_PATH + "/WESAD"
for sid in SUBJECT_IDS:
    print(sid)
    data_subject_indices.append(len(csv_data))
    print(data_subject_indices)
    subject = load_subject(f"{WESAD_path}/{sid}/{sid}.pkl")
    survey = load_survey(f"{WESAD_path}/{sid}/{sid}_readme.txt")

    labels = np.array(subject['label'])
    labels, av_labels = downsample_label(labels, SAMPLE_FS['LABEL'], 1.0/LABEL_WINDOW)

    bvp = np.array(subject['signal']['wrist']['BVP'])

    eda = np.array(subject['signal']['wrist']['EDA']).squeeze(1)

    for start in range(0, len(labels)):
        bvp_s = bvp[start * LABEL_WINDOW * SAMPLE_FS['BVP']: start * LABEL_WINDOW * SAMPLE_FS['BVP'] + INPUT_WINDOW * SAMPLE_FS['BVP']]
        eda_s = eda[start * LABEL_WINDOW * SAMPLE_FS['EDA']: start * LABEL_WINDOW * SAMPLE_FS['EDA'] + INPUT_WINDOW * SAMPLE_FS['EDA']]
        av_label_s = av_labels[start, :]

        ppg_features = {}
        ppg_features['ppg_mean'] = np.mean(bvp_s)
        ppg_features['ppg_std'] = np.std(bvp_s)
        ppg_features['ppg_skew'] = stats.skew(bvp_s)[0]
        ppg_features['ppg_kurtosis'] = stats.kurtosis(bvp_s)[0]
        
        # Heart rate variability features
        ppg_cleaned = nk.ppg_clean(bvp_s, sampling_rate=SAMPLE_FS['BVP'])
        signals, info = nk.ppg_process(ppg_cleaned, sampling_rate=SAMPLE_FS['BVP'])
        
        # Heart rate features
        ppg_features['hr_mean'] = np.mean(signals['PPG_Rate'])
        ppg_features['hr_std'] = np.std(signals['PPG_Rate'])
        
        # Additional PPG features
        hrv_features = nk.ppg_analyze(signals, sampling_rate=SAMPLE_FS['BVP'])
        if not hrv_features.empty:
            for col in hrv_features.columns:
                if not np.isnan(hrv_features[col].iloc[0]):
                    ppg_features[f'ppg_{col}'] = hrv_features[col].iloc[0]

        eda_features = {}
        eda_features['eda_mean'] = np.mean(eda_s)
        eda_features['eda_std'] = np.std(eda_s)
        eda_features['eda_skew'] = stats.skew(eda_s)
        eda_features['eda_kurtosis'] = stats.kurtosis(eda_s)
        
        # NeuroKit2 EDA analysis
        eda_cleaned = nk.eda_clean(eda_s, sampling_rate=SAMPLE_FS['EDA'])
        eda_decomposed = nk.eda_phasic(eda_cleaned, sampling_rate=SAMPLE_FS['EDA'])
        
        # Phasic and tonic components
        eda_features['eda_tonic_mean'] = np.mean(eda_decomposed['EDA_Tonic'])
        eda_features['eda_phasic_mean'] = np.mean(eda_decomposed['EDA_Phasic'])
        
        # Peak detection
        signals, info = nk.eda_peaks(eda_decomposed['EDA_Phasic'], sampling_rate=SAMPLE_FS['EDA'])
        eda_features['eda_peaks_count'] = len(info['SCR_Peaks'])

        label_dict = {"arousal_category" : av_label_s[0], "valence_category": av_label_s[1], "four_class": av_label_s[2]}

        merged = {**ppg_features, **eda_features, **label_dict}
        csv_data = pd.concat([csv_data, pd.DataFrame([merged])], ignore_index=True)
        
data_subject_indices.append(len(csv_data))
print(data_subject_indices)
csv_data.to_csv("output.csv", index=False)
    


S2
[0]
S3
[0, 3010]
S4
[0, 3010, 6228]
S5
[0, 3010, 6228, 9410]
S6
[0, 3010, 6228, 9410, 12510]
S7
[0, 3010, 6228, 9410, 12510, 16016]
S8
[0, 3010, 6228, 9410, 12510, 16016, 18606]
S9
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310]
S10
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892]
S11
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611]
S13
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611, 29199]
S14
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611, 29199, 31939]
S15
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611, 29199, 31939, 34684]
S16
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611, 29199, 31939, 34684, 37281]
S17
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611, 29199, 31939, 34684, 37281, 40067]
[0, 3010, 6228, 9410, 12510, 16016, 18606, 21310, 23892, 26611, 29199, 31939, 34684, 37281, 40067, 42998]


In [ ]:
# =========================== Data processing: Dapper ===================================================
SAMPLE_FS_Dapper = {'ACC': 20, 'PPG': 20, 'GSR': 40}
INPUT_WINDOW = 60
STEP_TIME = 6

dapper_path = "/home/general/Documents/Eric/Autonomous-Bio-Kinetic-Event-Detection/data/Dapper/"
pids = os.listdir(dapper_path + 'Physiol_Rec')

label_file = pd.read_excel(dapper_path + 'Psychol_Rec/ESM.xlsx')
label_file[" StartTime "] = pd.to_datetime(label_file[" StartTime "])

csv_data = pd.DataFrame() #pd.read_csv("output.csv")

for pid in pids:
    if pid == "README.txt":
        continue
    print(pid)
    # if int(pid) in csv_data["PID"].values:
    #     continue
    folder = dapper_path + f'Physiol_Rec/{pid}/'
    unique_dates = set()

    pid_label = label_file[label_file["Participant ID"] == int(pid)]

    for file in os.listdir(folder):
        file_arr = file.split('_')
        file_arr[-1] = file_arr[-1][:-4]
        if file_arr[0] in unique_dates:
            continue
        unique_dates.add(file_arr[0])

        if int(pid) == 3024 and file_arr[0] == "20191210220625":
            continue

        acc = pd.read_csv(folder + file_arr[0] + '_' + file_arr[1] + '_ACC.csv')
        acc = np.array(acc.drop(columns=['csv_time_motion']))
        
        gsr = pd.read_csv(folder + file_arr[0] + '_' + file_arr[1] + '_GSR.csv')
        gsr = np.array(gsr["GSR"])
        
        ppg = pd.read_csv(folder + file_arr[0] + '_' + file_arr[1] + '_PPG.csv')
        time = pd.to_datetime(ppg['csv_time_PPG'], format='mixed')
        ppg = np.array(ppg["PPG"])
        ppg_signals, ppg_info = nk.ppg_process(ppg, sampling_rate=SAMPLE_FS_Dapper['PPG'])

        for start in range(int((len(ppg) / SAMPLE_FS_Dapper['PPG'] - INPUT_WINDOW + STEP_TIME) // STEP_TIME)):
            ppg_s = ppg[start * STEP_TIME * SAMPLE_FS_Dapper['PPG']: start * STEP_TIME * SAMPLE_FS_Dapper['PPG'] + INPUT_WINDOW * SAMPLE_FS_Dapper['PPG']]
            peaks_s = np.array(ppg_signals['PPG_Peaks'][start * STEP_TIME * SAMPLE_FS_Dapper['PPG']: start * STEP_TIME * SAMPLE_FS_Dapper['PPG'] + INPUT_WINDOW * SAMPLE_FS_Dapper['PPG']])
            if np.count_nonzero(peaks_s) <= 2 or len(ppg_s) != INPUT_WINDOW * SAMPLE_FS_Dapper['PPG']:
                continue
            acc_s = acc[start * STEP_TIME * SAMPLE_FS_Dapper['ACC']: start * STEP_TIME * SAMPLE_FS_Dapper['ACC'] + INPUT_WINDOW * SAMPLE_FS_Dapper['ACC']]
            if acc_s.shape[0] != INPUT_WINDOW * SAMPLE_FS_Dapper['ACC']:
                continue
            acc_s = acc_s.reshape(100, 12, 3)

            gsr_s = gsr[start * STEP_TIME * SAMPLE_FS_Dapper['GSR']: start * STEP_TIME * SAMPLE_FS_Dapper['GSR'] + INPUT_WINDOW * SAMPLE_FS_Dapper['GSR']]
            if gsr_s.shape[0] != INPUT_WINDOW * SAMPLE_FS_Dapper['GSR']:
                continue
            acc_mean = np.mean(acc_s, axis = 0)
            acc_std = np.std(acc_s, axis = 0)
            acc_sum = np.trapezoid(np.abs(acc_s), axis = 0)
            acc_peak_freq = dom_nonzero_freq(acc_s, SAMPLE_FS_Dapper['ACC'])
            acc_data = np.concatenate([acc_mean.flatten(), acc_std.flatten(), acc_sum.flatten(), acc_peak_freq.flatten()])

            hrv = nk.hrv_time(peaks_s, sampling_rate=SAMPLE_FS_Dapper['PPG'], show=False) if not USE_FREQ_HRV else nk.hrv(peaks_s, sampling_rate=SAMPLE_FS_Dapper['PPG'], show=False)
            hr_mean = hrv['HRV_MeanNN']
            hr_std = hrv['HRV_SDNN']
            hrv_rmssd = hrv['HRV_RMSSD']
            hrv_pNN50 = hrv['HRV_pNN50']
            hrv_tinn = hrv['HRV_TINN']
            hrv_data = np.array([hr_mean, hr_std, hrv_rmssd, hrv_pNN50, hrv_tinn]).squeeze(1)

            if USE_FREQ_HRV:
                hrv_data.append(hrv['HRV_ULF'])
                hrv_data.append(hrv['HRV_LF'])
                hrv_data.append(hrv['HRV_HF'])
                hrv_data.append(hrv['HRV_VHF'])
                hrv_data.append(hrv['HRV_LFHF'])
                hrv_data.append(hrv['HRV_LFn'])
                hrv_data.append(hrv['HRV_HFn'])
            
            eda_feature, _, _ = eda_features(gsr_s, SAMPLE_FS_Dapper['GSR'])
            eda_data = [eda_feature["EDA_mean"], eda_feature["EDA_std"], eda_feature["EDA_max"], eda_feature["EDA_min"], eda_feature["EDA_slope"], eda_feature["EDA_range"], 
                        eda_feature["SCL_mean"], eda_feature["SCL_std"], eda_feature["SCR_mean"], eda_feature["SCR_std"], eda_feature["SCL_time_corr"], eda_feature["num_SCR_segments"],
                        eda_feature["sum_SCR_startle_magnitudes"], eda_feature["sum_response_durations_sec"], eda_feature["area_under_identified_SCR"]]

            merged = np.array(np.concatenate([acc_data, hrv_data, eda_data]))

            time_s = time.iloc[start * STEP_TIME * SAMPLE_FS_Dapper['PPG']]
            time_label = pid_label[time_s.floor('min') == pid_label[" StartTime "].dt.floor('min')]
            
            stress_label = 0
            if len(time_label) > 0 and ((time_label["PANAS_1"] >= 3).any() or (time_label["PANAS_6"] >= 3).any() or (time_label["PANAS_2"] >= 3).any() or (time_label["PANAS_4"] >= 3).any() or (time_label["PANAS_9"] >= 3).any()):
                stress_label = 1

            merged = {"PID": pid, "Features": merged, "Time": time_s, "Label": stress_label}

            csv_data = pd.concat([csv_data, pd.DataFrame([merged])], ignore_index=True)

    csv_data.to_parquet("Dapper_unnormed_6s.parquet", index=False)



1006
3013
3003
2025
2024
3029
1013
3026
1026
3023
3006
2030
1003
2001
2017
3019
1018
1012
1009
2012
2014
1011
1020
3117
2023
3008
1005
2016
2018
3024
1004
2020
2007
3004
2009
1008
2003
2005
1010
2019
3022
3002
3010
1001
3018
3005
1019
3007
2026
3001
2006
1007
1030
1024
1029
3027
2010
3009
1014
3020
1025
2027
3030
3011
3025
1027
3015
2004
2008
3128
2021
3028
3116
3014
1002
1021
2013
3012
2029
1028
1017
2022
2002
2028
2015
1022
1015
2011


In [16]:
# =========================== Data Normalization: Dapper ===================================================
extracted_data = pd.read_csv("output.csv")
all_features = extracted_data['Features'].str.strip('[]').str.split().apply(lambda x: [float(i) for i in x])
all_features = np.stack(all_features.values)

scaler = StandardScaler()
z_scores = scaler.fit_transform(all_features)

s = []
for id in extracted_data["PID"].unique():
    subject_data = extracted_data[extracted_data['PID'] == id]
    subject_data = subject_data['Features'].str.strip('[]').str.split().apply(lambda x: [float(i) for i in x])
    subject_data = np.stack(subject_data.values)
    subject_norm = scaler.fit_transform(subject_data)
    s.append(subject_norm)

subject_combined = np.concatenate(s, axis = 0)
norm_data = np.concatenate([z_scores, subject_combined], axis = 1)
print(norm_data.shape)

extracted_data["Features"] = list(norm_data)

(236731, 328)


In [20]:
extracted_data.to_parquet('norm_dapper.parquet')

In [3]:
csv_data = pd.read_csv("output.csv")

print(len(csv_data[csv_data["Label"] == 1]))
print(len(csv_data[csv_data["Label"] == 0]))

598
236133


In [52]:
np.save("EmoWear_X.npy", x)
np.save("EmoWear_Y.npy", y)

29